# Multi-camera person Re-ID: launch the platform

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbderrahimZineddineEsi/multi-modal-person-reid/blob/v2/notebooks/colab_demo.ipynb)

This notebook only **starts the app**. It does not process or evaluate anything. Run all cells
(**Runtime > Run all**, T4 GPU runtime), open the printed link, log in with the printed password,
then use the **Process** tab to upload a video or paste a video URL.

Your work lives on Google Drive (`MyDrive/reid_data/environments/<name>`, one folder per
environment), so it survives the runtime and can be synced with your PC: on the PC run
`reid sync push` before and `reid sync pull` after (see `docs/local_app.md`).
The benchmark walkthrough is in `colab_walkthrough.ipynb`.

## 1. GPU and installation

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os

REID_REF = "v2"  # branch or tag of the repository to install
os.environ["REID_REF"] = REID_REF

In [ ]:
%%bash
set -o pipefail
python -c "import torch; print('preinstalled torch', torch.__version__, '| CUDA', torch.version.cuda)"
pip install --progress-bar off "reid[models,app,eval] @ git+https://github.com/AbderrahimZineddineEsi/multi-modal-person-reid@${REID_REF}" \
  > /content/pip_install.log 2>&1 || { tail -n 40 /content/pip_install.log; exit 1; }
tail -n 3 /content/pip_install.log
python -c "import torch; print('torch after install', torch.__version__)"

`ultralytics` pulls in `opencv-python` and `insightface` pulls in `onnxruntime`, which clash with
`opencv-python-headless` and `onnxruntime-gpu`. The cell removes the clashing wheels and
reinstalls the two we want (harmless if nothing clashed). If a later cell fails with an
`ImportError` about `numpy` or `cv2`, use **Runtime > Restart session** and run again.

In [ ]:
%%bash
pip uninstall -y -q opencv-python onnxruntime
pip install -q --force-reinstall --no-deps opencv-python-headless "onnxruntime-gpu==1.32.0"
python -c "import cv2, onnxruntime; print('opencv', cv2.__version__, '| onnxruntime', onnxruntime.__version__)"

## 2. Settings and Drive

`PRESET` is the default model preset of the Process tab (`lite`, `balanced`, `accurate`, `max`).
`ENV` is the environment opened first; the app's top bar creates, switches and deletes others.

In [ ]:
from reid.notebook import sh

PRESET = "auto"  # auto = from the GPU (balanced on a T4) | lite | balanced | accurate | max
ENV = "default"  # environment opened at start (created if missing)
DRIVE_DATA_DIR = "/content/drive/MyDrive/reid_data"
LOCAL_DATA_DIR = "/content/reid_data"  # used if Drive is not mounted (lost with the runtime)

In [ ]:
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DATA_DIR)
except Exception as exc:  # not on Colab, or the prompt was declined
    DATA_DIR = Path(LOCAL_DATA_DIR)
    print(f"WARNING: Drive not mounted ({exc!r}).")
    print(f"Using {DATA_DIR}: on Colab everything is lost when this runtime ends.")
WORKSPACE = DATA_DIR / "environments"
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("workspace =", WORKSPACE)
sh(["reid", "env", "list", "--workspace", WORKSPACE])

## 3. Launch the app

`--share` opens a public `gradio.live` link, so a login is required: the cell generates a
password and prints it (add `--read-only` to hide the delete tools). The cell keeps running while
the app is up; stop it with the stop button. Before syncing to your PC, stop it so the database is
not being written.

In [ ]:
import secrets

PASSWORD = secrets.token_urlsafe(9)
print(f"Login: user 'reid', password '{PASSWORD}'")
sh(["reid", "app", "--share", "--auth", f"reid:{PASSWORD}", "--preset", PRESET,
    "--workspace", WORKSPACE, "--env", ENV])  # fmt: skip